# Part 2 — Improved, Strictly Following the DMorphNet Article
Gawade et al.'s D-MorphNet prescribes exactly one architecture: **EfficientNet-B6
(528², CLAHE, augmentation) → Global-Average-Pooled deep features → SVM (RBF)**, with a
transfer-learning stage where *"we fine-tune the model by making the top layers trainable
while keeping the earlier layers fixed"* (§3.4) and a threshold-optimization stage (§4.4).

This notebook improves Part 2 **without leaving that recipe**. Every step keeps the article's
architecture and is measured on the same frozen identity-disjoint test set (44 real +
44 morphs), so the effect of each refinement is a number:

| Step | Article anchor | Refinement |
|---|---|---|
| D1 | §4.1 | Part 2 baseline, reproduced |
| D2 | §3.5 | SVM-RBF hyper-parameters selected with *leakage-free* grouped CV |
| D3 | §3.1–3.2 | The article's data recipe at full strength: 400 morphs/class + *"the same steps for both real and morph images"* (balanced augmentation) |
| D4 | **§3.4 stage 2** | **Actually fine-tune EfficientNet-B6's top layers** (block7 + top conv, 13.7M params) on our data, then re-extract GAP features → SVM-RBF |
| D5 | §4.4 | Threshold optimization on grouped out-of-fold predictions (not an 80-image val set) |

Final section: the article's own evaluation artifacts (confusion matrix, ROC-AUC,
precision/recall/F1 table) for baseline vs improved, plus a robustness check.

In [ ]:
import os, sys, gc, json, time, random, zlib, warnings
from pathlib import Path
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import cv2
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from dmorphnet import data as D, evaluate as E, features as FE
from dmorphnet.config import RESULTS, DATASET, SEED, APP_MODEL, MODELS
from dmorphnet.preprocess import standardize

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix, roc_curve, auc,
                             classification_report)

random.seed(SEED); np.random.seed(SEED)
C_BLUE, C_ORANGE, C_GREEN, C_MAGENTA, C_RED = '#2a78d6', '#eb6834', '#008300', '#e87ba4', '#c62828'
plt.rcParams.update({'figure.dpi': 90, 'axes.spines.top': False, 'axes.spines.right': False})

steps = {}
def record(name, y, proba, threshold=0.5, note=''):
    m = E.basic_metrics(y, proba, threshold); m['eer'] = E.eer(y, proba); m['note'] = note
    steps[name] = m
    print(f"{name}:  acc {m['accuracy']:.3f} | P {m['precision']:.2f} | R {m['recall']:.2f} "
          f"| F1 {m['f1']:.3f} | AUC {m['auc']:.3f}  {note}")
    return m

def grouped_folds(Z, y, groups, n=5):
    return list(GroupKFold(n_splits=n).split(Z, y, groups))

def fit_calibrated(params, Z, y, folds):
    return CalibratedClassifierCV(SVC(kernel='rbf', random_state=SEED, **params),
                                  method='sigmoid', cv=folds, ensemble=False).fit(Z, y)

def grouped_oof(params, Z, y, groups, outer_folds):
    oof = np.zeros(len(y))
    for tr, te in outer_folds:
        inner = list(GroupKFold(3).split(Z[tr], y[tr], np.asarray(groups)[tr]))
        m = CalibratedClassifierCV(SVC(kernel='rbf', random_state=SEED, **params),
                                   method='sigmoid', cv=inner, ensemble=False).fit(Z[tr], y[tr])
        oof[te] = m.predict_proba(Z[te])[:, 1]
    return oof

RBF_GRID = {'C': [0.3, 1, 3, 10, 30, 100], 'gamma': ['scale', 3e-4, 1e-4, 3e-5, 1e-5]}

base = D.load_manifest(); D.assert_identity_disjoint(base)
y_te = np.array([0]*44 + [1]*44)
print('protocol ready — frozen test set: 44 real + 44 morph, unseen identities')

## D1 · Baseline — Part 2 as published
Same features, same `SVC(C=10, γ=10⁻⁴)` chosen by (leaked) naive CV in Part 2.

In [ ]:
z0 = np.load(RESULTS / 'features_b6.npz')
F_tr0, F_te0 = z0['X_train'], z0['X_test']
y_tr0 = np.array([0]*160 + [1]*160)
sc0 = StandardScaler().fit(F_tr0)
svm0 = SVC(kernel='rbf', C=10, gamma=1e-4, probability=True,
           random_state=SEED).fit(sc0.transform(F_tr0), y_tr0)
proba_d1 = svm0.predict_proba(sc0.transform(F_te0))[:, 1]
record('D1 · Part 2 baseline', y_te, proba_d1)

## D2 · Leakage-free SVM-RBF selection (article §3.5, done honestly)
The article's classifier stays: **SVM with RBF kernel**. Only the *selection protocol* changes —
`GroupKFold` by base image, so augmented copies can't vouch for themselves.

In [ ]:
def ids_only(files, target):
    names = [Path(f).name for f in files]
    return [names[i % len(names)] for i in range(target)]

rng = random.Random(SEED)
tr_files = {}
for cls in ('real', 'morph'):
    fs = base['train'][cls][:]
    rng.shuffle(fs)
    tr_files[cls] = fs[max(4, int(0.2 * len(fs))):]
g_tr0 = np.array(ids_only(tr_files['real'], 160) + ids_only(tr_files['morph'], 160))

Z_tr0, Z_te0 = sc0.transform(F_tr0), sc0.transform(F_te0)
folds0 = grouped_folds(Z_tr0, y_tr0, g_tr0)
gs_d2 = GridSearchCV(SVC(kernel='rbf', random_state=SEED), RBF_GRID,
                     cv=folds0, scoring='balanced_accuracy', n_jobs=-1).fit(Z_tr0, y_tr0)
print('honest RBF params:', gs_d2.best_params_, '| grouped CV %.3f' % gs_d2.best_score_)
cal_d2 = fit_calibrated(gs_d2.best_params_, Z_tr0, y_tr0, folds0)
proba_d2 = cal_d2.predict_proba(Z_te0)[:, 1]
record('D2 · honest SVM-RBF selection', y_te, proba_d2, note=str(gs_d2.best_params_))

## D3 · The article's data recipe at full strength (§3.1–3.2)
The article trains on a large balanced morph set and stresses *"we keep the same steps for both
real and morph images"*. Part 2 used only 130 of 741 possible train morph pairs and augmented
the two classes asymmetrically. Fixed here: **400 samples/class** with the augmentation chain
applied to *every* training image of *both* classes (features from the Part 3 labelled cache).

In [ ]:
z2 = np.load(RESULTS / 'features_v2_b6.npz', allow_pickle=True)
F_tr2, y_tr2, g_tr2 = z2['F_train'], z2['y_train'], np.array(z2['ids_train'])
F_te2 = z2['F_test']
sc2 = StandardScaler().fit(F_tr2)
Z_tr2, Z_te2 = sc2.transform(F_tr2), sc2.transform(F_te2)
folds2 = grouped_folds(Z_tr2, y_tr2, g_tr2)
gs_d3 = GridSearchCV(SVC(kernel='rbf', random_state=SEED), RBF_GRID,
                     cv=folds2, scoring='balanced_accuracy', n_jobs=-1).fit(Z_tr2, y_tr2)
print('params:', gs_d3.best_params_, '| grouped CV %.3f' % gs_d3.best_score_,
      f'(D2 was {gs_d2.best_score_:.3f} on the small set)')
cal_d3 = fit_calibrated(gs_d3.best_params_, Z_tr2, y_tr2, folds2)
proba_d3 = cal_d3.predict_proba(Z_te2)[:, 1]
record('D3 · full data recipe (400/class)', y_te, proba_d3, note=str(gs_d3.best_params_))

## D4 · §3.4 stage 2 — actually fine-tune EfficientNet-B6's top layers
The article: *"the model is used with fixed layers to extract the features … after that we
fine-tune the model by making the top layers trainable while keeping the earlier layers fixed."*
Part 2 only trained a dense head. Here the real thing, made CPU-feasible by **model surgery**:

* split B6 at the last block-6 layer (`block6k_add`, 17×17×344);
* run the frozen **bottom** once over every image and cache the activations;
* train the **top segment** (block7 + top conv — 13.7M params, Swish/BN intact) with a small
  sigmoid head on the cached activations (grouped validation split, early stopping);
* drop the head and re-extract **fine-tuned 2304-d GAP features → SVM-RBF**, exactly the
  article's hybrid, now with task-adapted top layers.

In [ ]:
import tensorflow as tf
from tensorflow import keras
tf.random.set_seed(SEED)

# rebuild the training/test images (deterministic seeds — must match the feature cache)
train_morphs_v2 = sorted((DATASET/'train'/'morph').glob('*.png'))
Xtr2, ytr_chk, ids_chk = D.build_split({'real': base['train']['real'], 'morph': train_morphs_v2},
                                       400, seed=SEED + zlib.crc32(b'train') % 1000, augment_all=True)
Xte2, yte_chk, _ = D.build_split(base['test'], 44, seed=SEED + zlib.crc32(b'test') % 1000)
assert np.array_equal(ytr_chk, y_tr2) and list(ids_chk) == list(g_tr2) and np.array_equal(yte_chk, y_te)

keras.backend.clear_session()
b6 = keras.applications.EfficientNetB6(include_top=False, weights='imagenet',
                                       pooling='avg', input_shape=(528, 528, 3))
CUT = [l.name for l in b6.layers if l.name.startswith('block6')][-1]
bottom = keras.Model(b6.input, b6.get_layer(CUT).output)
top = keras.Model(b6.get_layer(CUT).output, b6.output)      # shares layers with b6
bottom.trainable = False
print(f'cut at {CUT}: bottom -> {bottom.output_shape}, top {sum(np.prod(w.shape) for w in top.trainable_weights)/1e6:.1f}M trainable params')

def bottom_acts(X, batch=8):
    out = []
    for i in range(0, len(X), batch):
        out.append(bottom.predict(X[i:i+batch].astype(np.float32), verbose=0))
    return np.concatenate(out)

t0 = time.time()
A_tr = bottom_acts(Xtr2)
A_te = bottom_acts(Xte2)
del Xtr2, Xte2; gc.collect()
print(f'bottom activations cached: {A_tr.shape} + {A_te.shape} in {time.time()-t0:.0f}s')

In [ ]:
# fine-tune the top segment with a temporary sigmoid head (grouped val split)
tr_idx, va_idx = folds2[0]
inp = keras.Input(A_tr.shape[1:])
feat = top(inp)
out = keras.layers.Dense(1, activation='sigmoid')(keras.layers.Dropout(0.3)(feat))
ft = keras.Model(inp, out)
ft.compile(keras.optimizers.Adam(3e-5), 'binary_crossentropy', metrics=['accuracy'])
es = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)
t0 = time.time()
hist = ft.fit(A_tr[tr_idx], y_tr2[tr_idx], validation_data=(A_tr[va_idx], y_tr2[va_idx]),
              epochs=8, batch_size=16, callbacks=[es], verbose=0)
print(f'fine-tuned top layers in {time.time()-t0:.0f}s '
      f'({len(hist.history["loss"])} epochs, best val acc {max(hist.history["val_accuracy"]):.3f})')

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4))
ep = np.arange(1, len(hist.history['loss']) + 1)
for ax, a, ylab in [(axes[0], 'loss', 'binary cross-entropy'), (axes[1], 'accuracy', 'accuracy')]:
    ax.plot(ep, hist.history[a], color=C_BLUE, lw=2, label=f'train {a}')
    ax.plot(ep, hist.history[f'val_{a}'], color=C_ORANGE, lw=2, label=f'validation {a}')
    ax.set_xlabel('epoch'); ax.set_ylabel(ylab); ax.legend(frameon=False)
plt.suptitle('Fine-tuning EfficientNet-B6 top layers (block7 + top conv) — article §3.4 stage 2')
plt.tight_layout(); plt.savefig(RESULTS/'part2b_finetune_curves.png', bbox_inches='tight'); plt.show()

In [ ]:
# re-extract GAP features from the fine-tuned top -> the article's hybrid, upgraded
F_tr_ft = top.predict(A_tr, batch_size=16, verbose=0)
F_te_ft = top.predict(A_te, batch_size=16, verbose=0)
sc_ft = StandardScaler().fit(F_tr_ft)
Zft_tr, Zft_te = sc_ft.transform(F_tr_ft), sc_ft.transform(F_te_ft)
gs_d4 = GridSearchCV(SVC(kernel='rbf', random_state=SEED), RBF_GRID,
                     cv=folds2, scoring='balanced_accuracy', n_jobs=-1).fit(Zft_tr, y_tr2)
print('params:', gs_d4.best_params_, '| grouped CV %.3f' % gs_d4.best_score_,
      f'(frozen-feature D3 CV was {gs_d3.best_score_:.3f})')
cal_d4 = fit_calibrated(gs_d4.best_params_, Zft_tr, y_tr2, folds2)
proba_d4 = cal_d4.predict_proba(Zft_te)[:, 1]
record('D4 · fine-tuned B6 features + SVM', y_te, proba_d4, note=str(gs_d4.best_params_))

USE_FT = gs_d4.best_score_ >= gs_d3.best_score_
champ = dict(name='fine-tuned B6 + SVM-RBF' if USE_FT else 'frozen B6 + SVM-RBF (FT rejected by CV)',
             cal=cal_d4 if USE_FT else cal_d3,
             params=(gs_d4 if USE_FT else gs_d3).best_params_,
             Z_tr=Zft_tr if USE_FT else Z_tr2, Z_te=Zft_te if USE_FT else Z_te2,
             proba=proba_d4 if USE_FT else proba_d3,
             scaler=sc_ft if USE_FT else sc2)
print('champion by grouped CV:', champ['name'])

## D5 · Threshold optimization done right (article §4.4)
The article selects an operating threshold on a validation set; Part 2's 80-image val set made
that unstable. Here the threshold comes from **grouped out-of-fold predictions over all 800
training samples**.

In [ ]:
oof = grouped_oof(champ['params'], champ['Z_tr'], y_tr2, g_tr2, folds2)
ths = np.linspace(0.02, 0.98, 193)
bal = [balanced_accuracy_score(y_tr2, oof >= t) for t in ths]
t_bal = float(ths[int(np.argmax(bal))])
_, t_sec = E.bpcer_at_apcer(y_tr2, oof, 0.10)
print(f'thresholds — balanced {t_bal:.3f} | security(APCER<=10%) {t_sec:.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ths, bal, color=C_BLUE, lw=2, label='balanced accuracy (out-of-fold, n=800)')
ax.axvline(t_bal, color=C_GREEN, ls='--', lw=1.6, label=f'balanced threshold {t_bal:.2f}')
ax.axvline(t_sec, color=C_ORANGE, ls='--', lw=1.6, label=f'security threshold {t_sec:.2f}')
ax.set_xlabel('threshold on P(morph)'); ax.set_ylabel('score')
ax.set_title('Article §4.4 threshold optimization — on grouped OOF predictions')
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(RESULTS/'part2b_threshold.png', bbox_inches='tight'); plt.show()

m_d5 = record('D5 · champion @ optimized threshold', y_te, champ['proba'], threshold=t_bal,
              note=f't={t_bal:.2f}')

## Final evaluation — the article's own report format, baseline vs improved

In [ ]:
from sklearn.metrics import roc_auc_score
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, proba, ttl in [(axes[0], proba_d1, 'Part 2 baseline'),
                       (axes[1], champ['proba'], f"improved ({champ['name']})")]:
    pred = (proba >= (0.5 if ttl.startswith('Part') else t_bal)).astype(int)
    cm = confusion_matrix(y_te, pred)
    ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha='center', va='center', fontsize=15,
                color='white' if v > cm.max()/2 else '#0b0b0b')
    ax.set_xticks([0, 1], ['pred real', 'pred morph'])
    ax.set_yticks([0, 1], ['real', 'morph'])
    ax.set_title(f'Confusion — {ttl}')
for proba, col, lbl in [(proba_d1, C_ORANGE, 'baseline'), (champ['proba'], C_BLUE, 'improved')]:
    fpr, tpr, _ = roc_curve(y_te, proba)
    axes[2].plot(fpr, tpr, color=col, lw=2.2, label=f'{lbl} (AUC {auc(fpr, tpr):.3f})')
axes[2].plot([0, 1], [0, 1], '--', color='#9a9a9a', lw=1)
axes[2].set_xlabel('False Positive Rate'); axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('ROC — baseline vs improved'); axes[2].legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.savefig(RESULTS/'part2b_final_eval.png', bbox_inches='tight'); plt.show()

print(classification_report(y_te, (champ['proba'] >= t_bal).astype(int),
                            target_names=['real', 'morph']))
acc_lo, acc_hi = E.bootstrap_ci(y_te, champ['proba'],
                                lambda y, p: E.basic_metrics(y, p, t_bal)['accuracy'])
auc_lo, auc_hi = E.bootstrap_ci(y_te, champ['proba'], roc_auc_score)
print(f"accuracy {m_d5['accuracy']:.3f} (95% CI {acc_lo:.3f}–{acc_hi:.3f}) | "
      f"AUC {m_d5['auc']:.3f} (95% CI {auc_lo:.3f}–{auc_hi:.3f}) | EER {m_d5['eer']:.3f}")

In [ ]:
# robustness continuity check: the two other morph generators from earlier parts
spl = sorted((DATASET/'test'/'morph_spliced').glob('*.png'))
nai = sorted((DATASET/'test'/'morph_naive').glob('*.png'))
X_extra = np.stack([standardize(cv2.imread(str(f))) for f in spl + nai]).astype(np.uint8)
A_extra = bottom_acts(X_extra)
F_extra = top.predict(A_extra, batch_size=16, verbose=0) if USE_FT else FE.extract('b6', X_extra)
p_extra = champ['cal'].predict_proba(champ['scaler'].transform(F_extra))[:, 1]
p_spl, p_nai = p_extra[:len(spl)], p_extra[len(spl):]
print(f'APCER @ balanced threshold — spliced (unseen gen): {(p_spl < t_bal).mean():.3f} | '
      f'naive blend (unseen gen): {(p_nai < t_bal).mean():.3f}')
print('(both generators unseen by this article-faithful pipeline — the multi-generator training')
print(' variant from Part 4 remains the answer to this, at some cost on the seen generator)')

In [ ]:
# step-by-step summary chart
names = list(steps)
fig, ax = plt.subplots(figsize=(11.5, 4.6))
x = np.arange(len(names))
ax.plot(x, [steps[n]['accuracy'] for n in names], 'o-', color=C_BLUE, lw=2, label='accuracy')
ax.plot(x, [steps[n]['auc'] for n in names], 'o-', color=C_ORANGE, lw=2, label='ROC-AUC')
ax.plot(x, [steps[n]['f1'] for n in names], 'o-', color=C_GREEN, lw=2, label='F1 (morph)')
for i, n in enumerate(names):
    ax.annotate(f"{steps[n]['auc']:.3f}", (i, steps[n]['auc']),
                textcoords='offset points', xytext=(0, 8), fontsize=8, color=C_ORANGE)
ax.set_xticks(x, [n.split('·')[0].strip() for n in names])
ax.set_ylim(0.4, 1.05); ax.set_xlabel('refinement step (all EfficientNet-B6 + SVM-RBF)')
ax.set_title('Article-faithful refinement of Part 2 — one step, one measured effect')
ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(RESULTS/'part2b_step_progress.png', bbox_inches='tight'); plt.show()

for k, v in steps.items():
    print(f"{k:<42} acc {v['accuracy']:.3f}  P {v['precision']:.2f}  R {v['recall']:.2f} "
          f"F1 {v['f1']:.3f}  AUC {v['auc']:.3f}")

## Deployment artifact (v4) — the article's pipeline, refined

In [ ]:
import joblib
if USE_FT:
    top.save_weights(MODELS / 'b6_top_finetuned.weights.h5')
    wsize = (MODELS / 'b6_top_finetuned.weights.h5').stat().st_size / 1e6
    print(f'fine-tuned top weights saved ({wsize:.0f} MB)')

artifact = {
    'version': 4,
    'backbones': ['b6'],
    'scalers': {'b6': champ['scaler']},
    'model': champ['cal'],
    'classifier_family': 'SVC-RBF (per DMorphNet article)',
    'finetuned_top': 'models/b6_top_finetuned.weights.h5' if USE_FT else None,
    'cut_layer': CUT if USE_FT else None,
    'use_freq': False, 'use_tta': False,
    'threshold': t_bal, 'threshold_security': float(t_sec),
    'input_size': 528,
    'trained_on': f'{len(y_tr2)} samples (article recipe, 400/class, balanced augmentation)',
    'metrics_frozen_test': {k: {kk: float(vv) for kk, vv in v.items() if kk != 'note'}
                            for k, v in steps.items()},
}
joblib.dump(artifact, APP_MODEL / 'pipeline.joblib')

art = joblib.load(APP_MODEL / 'pipeline.joblib')
pv = art['model'].predict_proba(art['scalers']['b6'].transform(F_te_ft if USE_FT else F_te2))[:, 1]
assert np.allclose(pv, champ['proba'], atol=1e-9)
print('artifact v4 exported and reload-verified ✓')

with open(RESULTS / 'part2b_metrics.json', 'w') as fh:
    json.dump({'steps': {k: {kk: (float(vv) if not isinstance(vv, str) else vv)
                             for kk, vv in v.items()} for k, v in steps.items()},
               'champion': champ['name'], 'use_finetuned': bool(USE_FT),
               'cv': {'d2': float(gs_d2.best_score_), 'd3': float(gs_d3.best_score_),
                      'd4': float(gs_d4.best_score_)},
               'thresholds': {'balanced': t_bal, 'security': float(t_sec)},
               'acc_ci95': [acc_lo, acc_hi], 'auc_ci95': [auc_lo, auc_hi],
               'apcer_unseen': {'spliced': float((p_spl < t_bal).mean()),
                                'naive': float((p_nai < t_bal).mean())}}, fh, indent=2)
print('metrics written')

## Conclusions
*(filled after execution)*